# B9 - Simple StyleGAN2 Generator (no dataset needed)
- **Mapping network**: z(64) -> Dense64 LReLU -> Dense64 LReLU -> w(64)
- **Synthesis**: learned constant 4x4x32 -> ModConv32 (3x3) + noise -> Up 8x8 -> ModConv16 (3x3) + noise -> Up 16x16 -> Conv 1 filter Tanh, giving 16x16x1
- **Modulation**: $w'_{ijk} = s_i \cdot w_{ijk}$; **demodulation**: $w''_{ijk} = w'_{ijk} / \sqrt{\sum_{i,k} w'^2_{ijk} + \epsilon}$
- **Style mixing**: w from z1 feeds layer 1, w from z2 feeds layer 2

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
tf.keras.utils.set_random_seed(42)
LATENT = 64

class MappingNetwork(keras.Model):
    def __init__(self):
        super().__init__()
        self.net = keras.Sequential([layers.Dense(64), layers.LeakyReLU(0.2),
                                     layers.Dense(64), layers.LeakyReLU(0.2)])
    def call(self, z):
        z = z * tf.math.rsqrt(tf.reduce_mean(tf.square(z), axis=1, keepdims=True) + 1e-8)  # pixel norm
        return self.net(z)


class ModulatedConv2D(layers.Layer):
    def __init__(self, in_channels, filters, kernel_size=3, demodulate=True):
        super().__init__()
        self.demodulate = demodulate
        self.weight = self.add_weight(shape=(kernel_size, kernel_size, in_channels, filters),
                                      initializer='random_normal', name='w')
        self.affine = layers.Dense(in_channels, bias_initializer='ones')   # w -> style scales s

    def call(self, x, w):
        s = self.affine(w)                                       # (B, in_ch)
        # Modulation: scale each input channel of the kernel by s (done on the input = same result)
        x = x * s[:, None, None, :]
        y = tf.nn.conv2d(x, self.weight, strides=1, padding='SAME')
        if self.demodulate:
            w_mod = self.weight[None] * s[:, None, None, :, None]                          # (B,k,k,in,out)
            d = tf.math.rsqrt(tf.reduce_sum(tf.square(w_mod), axis=[1, 2, 3]) + 1e-8)      # (B, out)
            y = y * d[:, None, None, :]                                                    # demodulation
        return y


class NoiseInjection(layers.Layer):
    def __init__(self, channels):
        super().__init__()
        self.scale = self.add_weight(shape=(channels,), initializer=keras.initializers.Constant(0.1), name='noise_scale')
    def call(self, x):
        noise = tf.random.normal((tf.shape(x)[0], tf.shape(x)[1], tf.shape(x)[2], 1))
        return x + self.scale * noise


class Generator(keras.Model):
    def __init__(self):
        super().__init__()
        self.const = self.add_weight(shape=(1, 4, 4, 32), initializer='random_normal', name='const')
        self.conv1 = ModulatedConv2D(32, 32)
        self.noise1 = NoiseInjection(32)
        self.conv2 = ModulatedConv2D(32, 16)
        self.noise2 = NoiseInjection(16)
        self.up = layers.UpSampling2D(2)
        self.act = layers.LeakyReLU(0.2)
        self.to_img = layers.Conv2D(1, 3, padding='same', activation='tanh')

    def call(self, w1, w2=None):
        if w2 is None:            # no style mixing
            w2 = w1
        x = tf.tile(self.const, [tf.shape(w1)[0], 1, 1, 1])   # 4x4x32
        x = self.act(self.noise1(self.conv1(x, w1)))           # 4x4x32
        x = self.up(x)                                         # 8x8x32
        x = self.act(self.noise2(self.conv2(x, w2)))           # 8x8x16
        x = self.up(x)                                         # 16x16x16
        return self.to_img(x)                                  # 16x16x1

mapping = MappingNetwork()
generator = Generator()

In [ ]:
# Generate images from random latent vectors
z1 = tf.random.normal((6, LATENT))
z2 = tf.random.normal((6, LATENT))
w1, w2 = mapping(z1), mapping(z2)

img_a = generator(w1)            # style A only
img_b = generator(w2)            # style B only
img_mix = generator(w1, w2)      # style mixing: A at layer 1 (coarse), B at layer 2 (fine)
print("w shape:", w1.shape, "| image shape:", img_a.shape)

rows = [("Style A", img_a), ("Style B", img_b), ("Mixed A->B", img_mix)]
plt.figure(figsize=(10, 5))
for r, (name, imgs) in enumerate(rows):
    for i in range(6):
        plt.subplot(3, 6, r * 6 + i + 1); plt.imshow(imgs[i, :, :, 0], cmap='gray'); plt.axis('off')
        if i == 0: plt.title(name, loc='left')
plt.suptitle('Without style mixing (rows 1-2) vs with style mixing (row 3)'); plt.show()

In [ ]:
# Effect of noise injection: same w, two forward passes -> different fine details
a, b = generator(w1[:1]), generator(w1[:1])
plt.figure(figsize=(8, 3))
for i, (im, t) in enumerate([(a, 'pass 1'), (b, 'pass 2'), (tf.abs(a - b), '|difference|')]):
    plt.subplot(1, 3, i + 1); plt.imshow(im[0, :, :, 0], cmap='gray'); plt.title(t); plt.axis('off')
plt.show()

print("Mean abs difference between with-mixing and without-mixing images:",
      float(tf.reduce_mean(tf.abs(img_mix - img_a))))

**Observation:** the generator is untrained, so the images are abstract patterns, but the architecture behaviour is visible:
- Different z give different images.
- Mixed images take their overall layout from style A and their finer texture from style B.
- Noise injection changes only small local details for the same w.